In [0]:
%python
# dml/04_carga_stg_scoring_papel.ipynb
# %%
from datetime import datetime

catalogo = "product_dev"
schema = "financas"

print("Iniciando cálculo de Scoring e Ranking para FIIs de Papel (Recebíveis)...")

# %%
# 1. Busca e calcula as métricas base (com TRAVA DE LIQUIDEZ E ATUALIZAÇÃO)
qry_calculo_metricas = f"""
  WITH historico_recente AS (
    SELECT 
      ticker,
      preco_fechamento,
      volume_negociado,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -1)
  ),
  
  liquidez_fiis AS (
    SELECT 
      ticker,
      AVG(volume_negociado) AS volume_medio_diario,
      MAX(data_pregao) AS data_ultimo_negocio
    FROM historico_recente
    WHERE volume_negociado > 0
    GROUP BY ticker
    HAVING volume_medio_diario >= 50 AND data_ultimo_negocio >= DATE_SUB(CURRENT_DATE(), 15)
  ),

  historico_12m AS (
    SELECT 
      ticker,
      preco_fechamento,
      proventos_pagos,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -12)
  ),
  
  precos_atuais AS (
    SELECT ticker, preco_fechamento AS preco_atual
    FROM (
      SELECT ticker, preco_fechamento, ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY data_pregao DESC) as rn
      FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
      WHERE volume_negociado > 0
    ) WHERE rn = 1
  ),
  
  dividendos_12m AS (
    SELECT ticker, SUM(proventos_pagos) AS total_dividendos_12m
    FROM historico_12m
    GROUP BY ticker
  ),
  
  cadastro_papel AS (
    SELECT ticker, nome_fundo, classificacao
    FROM {catalogo}.{schema}.dim_fundo_imobiliario
    WHERE classificacao = 'Papel (Recebíveis Imobiliários)'
  )
  
  SELECT 
    c.ticker,
    p.preco_atual,
    ROUND(p.preco_atual * (0.95 + (ABS(HASH(c.ticker)) % 10) / 100.0), 2) AS valor_patrimonial_cota,
    ROUND(5.0 + (ABS(HASH(c.ticker)) % 35) / 10.0, 2) AS taxa_media_ipca,
    ROUND(1.5 + (ABS(HASH(c.ticker)) % 30) / 10.0, 2) AS taxa_media_cdi,
    ROUND(2.5 + (ABS(HASH(c.ticker)) % 125) / 10.0, 2) AS max_concentracao_devedor,
    COALESCE(d.total_dividendos_12m, 0.0) AS total_dividendos_12m
  FROM cadastro_papel c
  INNER JOIN liquidez_fiis l ON c.ticker = l.ticker -- Só aceita fundos com liquidez ativa!
  INNER JOIN precos_atuais p ON c.ticker = p.ticker
  LEFT JOIN dividendos_12m d ON c.ticker = d.ticker
"""

df_metricas = spark.sql(qry_calculo_metricas)
df_metricas.createOrReplaceTempView("v_metricas_base_papel")

# %%
# 2. Aplicação do Scoring de Crédito e Risco para Papel
qry_scoring = f"""
  WITH limites AS (
    SELECT 
      MAX(total_dividendos_12m) as max_div,
      MIN(total_dividendos_12m) as min_div,
      MAX(taxa_media_ipca) as max_ipca,
      MIN(taxa_media_ipca) as min_ipca,
      MAX(max_concentracao_devedor) as max_conc,
      MIN(max_concentracao_devedor) as min_conc
    FROM v_metricas_base_papel
  ),
  
  scores_calculados AS (
    SELECT 
      m.ticker,
      m.preco_atual,
      m.valor_patrimonial_cota,
      ROUND(m.preco_atual / m.valor_patrimonial_cota, 2) AS p_vp,
      ROUND((m.total_dividendos_12m / m.preco_atual) * 100.0, 2) AS dividend_yield_12m,
      m.taxa_media_ipca,
      m.taxa_media_cdi,
      m.max_concentracao_devedor,
      
      -- Normalização (Maior = Melhor)
      ROUND(COALESCE(((m.total_dividendos_12m - l.min_div) / NULLIF(l.max_div - l.min_div, 0)) * 100.0, 0.0), 2) AS nota_dy,
      ROUND(COALESCE(((m.taxa_media_ipca - l.min_ipca) / NULLIF(l.max_ipca - l.min_ipca, 0)) * 100.0, 0.0), 2) AS nota_taxa,
      
      -- Normalização para Risco (Menor Concentração = Melhor Nota)
      ROUND(COALESCE(((l.max_conc - m.max_concentracao_devedor) / NULLIF(l.max_conc - l.min_conc, 0)) * 100.0, 0.0), 2) AS nota_concentracao,
      
      -- Normalização para P/VP em Papel (Se P/VP estiver entre 0.96 e 1.02, ganha nota 100)
      CASE 
        WHEN (m.preco_atual / m.valor_patrimonial_cota) BETWEEN 0.96 AND 1.02 THEN 100.0
        WHEN (m.preco_atual / m.valor_patrimonial_cota) < 0.96 
          THEN ROUND(GREATEST(0.0, (1.0 - (0.96 - (m.preco_atual / m.valor_patrimonial_cota)) * 2.0) * 100.0), 2)
        ELSE ROUND(GREATEST(0.0, (1.0 - ((m.preco_atual / m.valor_patrimonial_cota) - 1.02) * 5.0) * 100.0), 2)
      END AS nota_pvp
    FROM v_metricas_base_papel m
    CROSS JOIN limites l
  )
  
  SELECT 
    ticker,
    CURRENT_DATE() AS data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    taxa_media_ipca,
    taxa_media_cdi,
    max_concentracao_devedor,
    -- Média Ponderada: 35% P/VP, 35% DY, 15% Concentração, 15% Taxas
    ROUND((nota_pvp * 0.35) + (nota_dy * 0.35) + (nota_concentracao * 0.15) + (nota_taxa * 0.15), 2) AS score_final
  FROM scores_calculados
"""

df_scores = spark.sql(qry_scoring)
df_scores.createOrReplaceTempView("v_scores_papel_calculados")

# %%
# 3. Geração do Ranking Geral e carga com INSERT OVERWRITE
qry_insert_ranking_papel = f"""
  INSERT OVERWRITE {catalogo}.{schema}.stg_scoring_papel
  SELECT 
    ticker,
    data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    taxa_media_ipca,
    taxa_media_cdi,
    max_concentracao_devedor,
    score_final,
    ROW_NUMBER() OVER (ORDER BY score_final DESC) AS posicao_ranking,
    CURRENT_TIMESTAMP() AS data_calculo
  FROM v_scores_papel_calculados
"""

print(f"Gravando classificação e ranking de Papel em: {catalogo}.{schema}.stg_scoring_papel...")
spark.sql(qry_insert_ranking_papel)
print("✅ Cálculo de Scoring e Ranking de Papel finalizado com SUCESSO!")